# Capstone — the paper, mirrored

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**The paper is live at [louai-mohamed.github.io/flyrank-ml-internship](https://louai-mohamed.github.io/flyrank-ml-internship/).**
This notebook is the paper's engine room: it reads the four committed receipts, rebuilds every figure the page
shows, and then audits the page itself — section by section, number by number.

It needs no warehouse access and no token. If the page and the receipts ever disagree, this notebook fails.


## 1. Question

*The research question and the decision it supports.*

A portfolio of content pages outgrows the people who maintain it. The decision is narrow: **which fifty pages
does an editor open this week?** The question behind it is measurable: given what was observable on 31 March
2026, which pages lost search clicks in April — and does a model order them better than the frozen rule a
practitioner would use without one?

The unit is one content page at one decision moment. The answer has to be honest in two directions: good enough
to order a review queue, never strong enough to justify changing a page automatically.


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

ROOT = Path.cwd()
while not (ROOT / "work" / "outputs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
OUT = ROOT / "work" / "outputs"
DOCS = ROOT / "docs"
ASSETS = DOCS / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)
PAPER = DOCS / "index.html"

def receipt(name):
    return json.loads((OUT / name).read_text(encoding="utf-8"))

base = receipt("w04_baseline_metrics.json")     # signals, frozen rule, first queue
model = receipt("w05_model_metrics.json")       # features, split, model comparison
audit = receipt("w06_validation_audit.json")    # split audit, leakage injection
play = receipt("w07_playbook_summary.json")     # reason codes, gates, archetypes, batch

print("repo root :", ROOT)
print("receipts  :", ", ".join(p.name for p in sorted(OUT.glob("*.json"))))
print("paper     :", PAPER.exists(), f"({PAPER.stat().st_size:,} bytes)" if PAPER.exists() else "")

# ── formatting helpers: one place that decides how a number is written ──────
def pct(x, nd=1):
    """0.5148 -> '51.5%'; 0.96 -> '96%' (no trailing .0)."""
    s = f"{x * 100:.{nd}f}%"
    return s.replace(".0%", "%")

def p3(x):
    """Precision@K: at most three decimals, no trailing zeros (0.820 -> '0.82')."""
    return f"{x:.3f}".rstrip("0").rstrip(".")

def auc(x):
    """Rank metrics keep three decimals so 1.0 reads as '1.000' next to 0.617."""
    return f"{x:.3f}"

def th(x):
    return f"{int(round(x)):,}"

def rng(lo, hi):
    return f"{lo:.2f}\u2013{hi:.2f}"

# ── the page palette, so figures and page are one artifact ─────────────────
INK, INK2, INK3 = "#17181a", "#4e5057", "#7c7e86"
PAPER_C, PANEL, RULE = "#fbfaf7", "#f4f2ec", "#e2ded4"
ACCENT, ACCENT2, CLAY, OKC = "#1d4e89", "#2f6fb3", "#9c5228", "#2f6b4f"

plt.rcParams.update({
    "figure.facecolor": PAPER_C, "axes.facecolor": PAPER_C, "savefig.facecolor": PAPER_C,
    "font.family": "DejaVu Sans", "font.size": 10,
    "axes.edgecolor": RULE, "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.grid": True, "grid.color": RULE, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})

def head(ax, title, sub=None):
    """Left-aligned title + a quieter subtitle, the way the page reads."""
    ax.set_title(title, loc="left", fontsize=12.5, fontweight="bold", pad=18 if sub else 10)
    if sub:
        ax.annotate(sub, xy=(0, 1.015), xycoords="axes fraction", ha="left", va="bottom",
                    fontsize=9.5, color=INK2)

def save(fig, name):
    path = ASSETS / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {path.relative_to(ROOT).as_posix()}  ({path.stat().st_size:,} bytes)")
    return path


repo root : E:\AI\PE\flyrank-ml-internship
receipts  : w04_baseline_metrics.json, w05_model_metrics.json, w06_validation_audit.json, w07_playbook_summary.json
paper     : True (84,867 bytes)


In [2]:
# Every number the page states as a fact is generated here, from a receipt.
# The page carries the same keys as `data-fact` attributes; §7 checks them.
guide = (ROOT / "docs" / "ml-intern-dataset-and-lane-guide.md").read_text(encoding="utf-8")
fact_rows = int(re.search(r"\|\s*daily fact rows\s*\|\s*([\d,]+)\s*\|", guide).group(1).replace(",", ""))

fr, arch = play["frame"], play["archetypes"]
dec, wb = play["decay_evidence"], play["weekly_batch"]
m = model["metrics"]
sc, inj = audit["split_comparison"], audit["leakage"]["injection"]
sig = base["signals"]
mon = {row["metric"]: row["current"] for row in play["monitoring"]}

# the share of the queue whose position tier has no peer CTR median (deep tiers)
total_pages = sum(a["pages"] for a in arch.values())
sized_pages = sum(a["pages"] * a["share_sized"] for a in arch.values())

FACTS = {
    # frame
    "march_pages":        th(fr["march_pages_in_release"]),
    "frame_pages":        th(fr["pages_described_by_features"]),
    "frame_share":        pct(fr["share_of_march_pages_covered"]),
    "evaluable_pages":    th(fr["labelled_rows_used_for_refit"]),
    "dropped_no_april":   th(fr["scored_but_never_validated"]),
    "clients":            str(fr["clients_in_label"]),
    "queue_clients":      str(fr["clients_in_queue"]),
    "base_rate":          pct(fr["observed_decline_base_rate"]),
    "fact_rows":          f"{fact_rows / 1e6:.1f} million",
    "unsized_share":      pct(1 - sized_pages / total_pages),
    # the two signal verdicts that reframed the problem
    "ctr_spread":         f"{sig['ctr_vs_position_peers']['spread']:.3f}",
    "ctr_n":              th(sig['ctr_vs_position_peers']['n_visible']),
    "fresh_decline":      pct(sig["content_age_staleness_proxy"]["fresh_decline"]),
    "old_decline":        pct(sig["content_age_staleness_proxy"]["oldest_bucket_decline"]),
    "small_decline":      pct(sig["volume_quick_win"]["small_bucket_decline"]),
    "large_decline":      pct(sig["volume_quick_win"]["large_bucket_decline"]),
    # split honesty
    "random_p50":         p3(sc["before"]["P@50"]),
    "random_auc":         auc(sc["before"]["AUC"]),
    "grouped_p50":        p3(sc["after"]["P@50"]),
    "grouped_auc":        auc(sc["after"]["AUC"]),
    "icc":                f"{sc['label_icc_across_clients']:.3f}",
    # model vs baseline
    "rule_p50":           p3(m["baseline"]["p_at_50_mean"]),
    "rule_p50_range":     rng(m["baseline"]["p_at_50_min"], m["baseline"]["p_at_50_max"]),
    "rule_auc":           auc(m["baseline"]["roc_auc_mean"]),
    "logistic_p50":       p3(m["logistic"]["p_at_50_mean"]),
    "hgb_p10":            p3(m["hist-gbm"]["p_at_10_mean"]),
    "hgb_p20":            p3(m["hist-gbm"]["p_at_20_mean"]),
    "hgb_p50":            p3(m["hist-gbm"]["p_at_50_mean"]),
    "hgb_p50_range":      rng(m["hist-gbm"]["p_at_50_min"], m["hist-gbm"]["p_at_50_max"]),
    "hgb_p100":           p3(m["hist-gbm"]["p_at_100_mean"]),
    "hgb_auc":            auc(m["hist-gbm"]["roc_auc_mean"]),
    "hgb_pr":             auc(m["hist-gbm"]["pr_auc_mean"]),
    "hgb_nomom_p50":      p3(m["hist-gbm (no momentum)"]["p_at_50_mean"]),
    "feature_share":      f"{model['gain_decomposition_p_at_50']['share_of_gain_from_feature'] * 100:.0f}%",
    "perm_momentum":      f"{model['top_features_permutation_auc_drop']['log_momentum']:.3f}",
    "perm_ctr":           f"{model['top_features_permutation_auc_drop']['ctr_ratio']:.3f}",
    "perm_impressions":   f"{model['top_features_permutation_auc_drop']['log_impressions']:.3f}",
    # leakage injection
    "inj_shipped_auc":    auc(inj["shipped features"]["AUC"]),
    "inj_april_auc":      auc(inj["+ imp_apr (the label's own ingredient)"]["AUC"]),
    "inj_ratio_auc":      auc(inj["+ april_ratio (near-label)"]["AUC"]),
    "ablation_auc":       auc(inj["- log_momentum (ablation)"]["AUC"]),
    # playbook
    "actionable":         th(play["dispositions"]["actionable"]),
    "watchlist":          th(play["dispositions"]["watchlist"]),
    "no_action":          th(play["dispositions"]["no_action"]),
    "actionable_share":   pct(play["actionable_share_of_queue"]),
    "client_concentration": pct(mon["top-100 client concentration"], nd=0),
    "low_expected_share": pct(mon["share of visible pages below 5 expected clicks"]),
    "fading_rate":        pct(dec["fading_decline_rate"]),
    "steady_rate":        pct(dec["steady_decline_rate"]),
    "both_rate":          pct(dec["both_signals_decline_rate"]),
    "neither_rate":       pct(dec["neither_signal_decline_rate"]),
    "batch_actions":      str(wb["actions"]),
    "batch_clients":      str(wb["clients"]),
    "batch_hours":        f"{wb['hours']:.0f}",
    "batch_clicks":       f"{wb['measured_click_gap']:.0f}",
}

print(f"{len(FACTS)} facts, generated from four receipts + the dataset guide\n")
for k, v in FACTS.items():
    print(f"  {k:22} {v}")


55 facts, generated from four receipts + the dataset guide

  march_pages            176,738
  frame_pages            101,441
  frame_share            57.4%
  evaluable_pages        100,893
  dropped_no_april       548
  clients                43
  queue_clients          44
  base_rate              51.5%
  fact_rows              78.8 million
  unsized_share          24.5%
  ctr_spread             0.212
  ctr_n                  76,131
  fresh_decline          42.1%
  old_decline            49.9%
  small_decline          53.5%
  large_decline          44.5%
  random_p50             0.992
  random_auc             0.791
  grouped_p50            0.82
  grouped_auc            0.643
  icc                    0.133
  rule_p50               0.54
  rule_p50_range         0.46–0.68
  rule_auc               0.554
  logistic_p50           0.748
  hgb_p10                0.84
  hgb_p20                0.79
  hgb_p50                0.82
  hgb_p50_range          0.62–0.94
  hgb_p100               0.806
 

## 2. Data

*Which release, which tables, date windows, what was excluded and why. Public-safe.*

One fixed snapshot of a pseudonymized production search warehouse
(`flyrank_pseudonymized_warehouse_release_v20260703`, export 2026-07-03, daily facts stop 2026-06-30).
Features come from March 2026, the label from April 2026, and everything after April was never read.

The frame is the honest part of this section: three filters decide who is in the study, and each one drops
someone worth naming.


In [3]:
frame_tbl = pd.DataFrame([
    ("March pages with any impressions", th(fr["march_pages_in_release"]),
     "the whole release, before any filter"),
    (">= 100 March impressions", th(fr["pages_described_by_features"]),
     f"{FACTS['frame_share']} of March — the study frame"),
    ("with an April row (labelled)", th(fr["labelled_rows_used_for_refit"]),
     f"{FACTS['dropped_no_april']} pages dropped: no April activity"),
    ("clients in the label", FACTS["clients"], "the grouped split operates on these"),
    ("clients in the queue", FACTS["queue_clients"], "one more than the label: scored, never validated"),
], columns=["stage", "pages / clients", "what it means"])

print("THE FRAME\n")
print(frame_tbl.to_string(index=False))

print(f"\nfact rows in the release: {fact_rows:,}  ({FACTS['fact_rows']} daily rows)")
print(f"pages whose position tier has no peer CTR median: {FACTS['unsized_share']} of the queue")
print("\nEXCLUDED, AND WHY")
for col, why in [
    ("imp_apr, declined", "April measurements — they define the label"),
    ("content_updated_date", "future-dated on most rows; unusable as age"),
    ("baseline_score, ctr_points, clicks_lost", "products of the rule baseline being compared against"),
    ("ctr_ratio at deep positions", "no peer median exists, so no click gap can be sized"),
]:
    print(f"  - {col:42} {why}")

print("\nSIGNAL AUDIT (week 4, verdicts carried into this paper)")
for name, s in sig.items():
    if name == "ctr_vs_position_peers":
        detail = f"spread {s['spread']:.3f} over {th(s['n_visible'])} visible pages"
    elif name == "content_age_staleness_proxy":
        detail = f"fresh {pct(s['fresh_decline'])} vs oldest {pct(s['oldest_bucket_decline'])}"
    else:
        detail = f"small {pct(s['small_bucket_decline'])} vs large {pct(s['large_bucket_decline'])}"
    print(f"  {name:26} {s['verdict']:9} {detail}")


THE FRAME

                           stage pages / clients                                    what it means
March pages with any impressions         176,738             the whole release, before any filter
        >= 100 March impressions         101,441                 57.4% of March — the study frame
    with an April row (labelled)         100,893             548 pages dropped: no April activity
            clients in the label              43              the grouped split operates on these
            clients in the queue              44 one more than the label: scored, never validated

fact rows in the release: 78,835,655  (78.8 million daily rows)
pages whose position tier has no peer CTR median: 24.5% of the queue

EXCLUDED, AND WHY
  - imp_apr, declined                          April measurements — they define the label
  - content_updated_date                       future-dated on most rows; unusable as age
  - baseline_score, ctr_points, clicks_lost    products of the rule 

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Three decisions carry the paper, and each one was made against the easier alternative:

1. **The label is traffic loss, not quality.** `declined` = April impressions below 80% of March. Blunt,
   reproducible from two aggregates, and honest about what it measures.
2. **The split holds clients out.** A random page split lets a model memorise client identity and report it as
   skill — the audit measures exactly that, and the difference is the paper's spine.
3. **The baseline is frozen before the model is trained.** It is the rule a practitioner would use without a
   model, not a straw man tuned to lose.


In [4]:
features = pd.DataFrame([
    ("ctr_ratio", "numeric", "page CTR / median CTR of its position tier"),
    ("zero_clicks", "numeric", "no clicks in March despite impressions"),
    ("log_impressions", "numeric", "March impressions, logged"),
    ("avg_position_30d", "numeric", "mean rank across March"),
    ("days_with_impressions_30d", "numeric", "distinct days with any impression"),
    ("content_age_days", "numeric", "age at 2026-03-31 (context only, never a trigger)"),
    ("log_momentum", "numeric", "log(last-half / first-half March impressions)"),
    ("content_type", "categorical", "page type"),
], columns=["feature", "kind", "what it measures"])
print("FEATURES (all knowable on 2026-03-31)\n")
print(features.to_string(index=False))

split_tbl = pd.DataFrame([
    ("design before", sc["design_before"], p3(sc["before"]["P@50"]), auc(sc["before"]["AUC"])),
    ("design after", sc["design_after"], p3(sc["after"]["P@50"]), auc(sc["after"]["AUC"])),
], columns=["split", "how", "P@50", "ROC-AUC"])
print("\n\nSPLIT DESIGN\n")
print(split_tbl.to_string(index=False))
print(f"\nlabel ICC across clients: {FACTS['icc']}  — a random page split reports client memorisation as skill")
print(f"seeds: {', '.join(str(s) for s in model['split']['seeds'])}; "
      f"{model['split']['test_clients_per_fold']} clients held out per draw")

leak_tbl = pd.DataFrame([
    ("shipped features", auc(inj["shipped features"]["AUC"]), p3(inj["shipped features"]["P@50"]), "honest reference"),
    ("+ imp_apr", auc(inj["+ imp_apr (the label's own ingredient)"]["AUC"]),
     p3(inj["+ imp_apr (the label's own ingredient)"]["P@50"]), "the label's own ingredient"),
    ("+ april_ratio", auc(inj["+ april_ratio (near-label)"]["AUC"]),
     p3(inj["+ april_ratio (near-label)"]["P@50"]), "a near-label"),
    ("- log_momentum", auc(inj["- log_momentum (ablation)"]["AUC"]),
     p3(inj["- log_momentum (ablation)"]["P@50"]), "ablation"),
], columns=["feature set", "ROC-AUC", "P@50", "what it is"])
print("\n\nLEAKAGE INJECTION (grouped split)\n")
print(leak_tbl.to_string(index=False))
print("\nverdict:", audit["leakage"]["verdict"])
print("feature query touches April:", audit["leakage"]["feature_query_touches_april"],
      "| product flags present:", audit["leakage"]["product_flag_columns_present"] or "none")


FEATURES (all knowable on 2026-03-31)

                  feature        kind                                  what it measures
                ctr_ratio     numeric        page CTR / median CTR of its position tier
              zero_clicks     numeric            no clicks in March despite impressions
          log_impressions     numeric                         March impressions, logged
         avg_position_30d     numeric                            mean rank across March
days_with_impressions_30d     numeric                 distinct days with any impression
         content_age_days     numeric age at 2026-03-31 (context only, never a trigger)
             log_momentum     numeric     log(last-half / first-half March impressions)
             content_type categorical                                         page type


SPLIT DESIGN

        split                                                                      how  P@50 ROC-AUC
design before                        random page spl

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The comparison is deliberately lopsided in one direction: the baseline is the rule from week 4, frozen before
the model existed. Everything below is the mean across five client-grouped draws, with the min–max range shown
where the spread matters.


In [5]:
res = pd.DataFrame([
    ("Frozen rule baseline", p3(m["baseline"]["p_at_10_mean"]), p3(m["baseline"]["p_at_20_mean"]),
     p3(m["baseline"]["p_at_50_mean"]), p3(m["baseline"]["p_at_100_mean"]),
     auc(m["baseline"]["roc_auc_mean"]), auc(m["baseline"]["pr_auc_mean"])),
    ("Logistic regression", p3(m["logistic"]["p_at_10_mean"]), p3(m["logistic"]["p_at_20_mean"]),
     p3(m["logistic"]["p_at_50_mean"]), p3(m["logistic"]["p_at_100_mean"]),
     auc(m["logistic"]["roc_auc_mean"]), auc(m["logistic"]["pr_auc_mean"])),
    ("Gradient boosting (shipped)", p3(m["hist-gbm"]["p_at_10_mean"]), p3(m["hist-gbm"]["p_at_20_mean"]),
     p3(m["hist-gbm"]["p_at_50_mean"]), p3(m["hist-gbm"]["p_at_100_mean"]),
     auc(m["hist-gbm"]["roc_auc_mean"]), auc(m["hist-gbm"]["pr_auc_mean"])),
    ("Gradient boosting, no momentum", p3(m["hist-gbm (no momentum)"]["p_at_10_mean"]),
     p3(m["hist-gbm (no momentum)"]["p_at_20_mean"]), p3(m["hist-gbm (no momentum)"]["p_at_50_mean"]),
     p3(m["hist-gbm (no momentum)"]["p_at_100_mean"]),
     auc(m["hist-gbm (no momentum)"]["roc_auc_mean"]), auc(m["hist-gbm (no momentum)"]["pr_auc_mean"])),
], columns=["ranker", "P@10", "P@20", "P@50", "P@100", "ROC-AUC", "PR-AUC"])

print(f"CLIENT-GROUPED HOLDOUTS — base rate {FACTS['base_rate']}, 5 seeds\n")
print(res.to_string(index=False))
print(f"\nP@50 range: shipped {FACTS['hgb_p50_range']} | rule {FACTS['rule_p50_range']}")
print(f"share of the lift from the momentum feature: {FACTS['feature_share']}")
print(f"ablation without log_momentum: P@50 {FACTS['hgb_nomom_p50']} — below the rule baseline it must beat")

# ── figure 1: precision at K ───────────────────────────────────────────────
ks = [10, 20, 50, 100]
def curve(key):
    return [m[key][f"p_at_{k}_mean"] for k in ks]

fig, ax = plt.subplots(figsize=(9.4, 4.7))
ax.plot(ks, curve("baseline"), color=INK3, lw=1.8, ls="--", marker="o", ms=5, label="Frozen rule baseline")
ax.plot(ks, curve("logistic"), color=ACCENT2, lw=1.8, marker="o", ms=5, label="Logistic regression")
ax.plot(ks, curve("hist-gbm (no momentum)"), color=CLAY, lw=1.6, ls=":", marker="s", ms=4.5,
        label="Gradient boosting, no momentum")
ax.plot(ks, curve("hist-gbm"), color=ACCENT, lw=2.8, marker="o", ms=7, label="Gradient boosting (shipped)")

# the only ranges we have are at K = 50 — draw them there and nowhere else
for key, colour, lo, hi in [("hist-gbm", ACCENT, m["hist-gbm"]["p_at_50_min"], m["hist-gbm"]["p_at_50_max"]),
                            ("baseline", INK3, m["baseline"]["p_at_50_min"], m["baseline"]["p_at_50_max"])]:
    ax.errorbar([50], [m[key]["p_at_50_mean"]], yerr=[[m[key]["p_at_50_mean"] - lo], [hi - m[key]["p_at_50_mean"]]],
                fmt="none", ecolor=colour, elinewidth=1.6, capsize=4, alpha=0.75, zorder=1)

ax.axhline(fr["observed_decline_base_rate"], color=INK3, lw=1, ls=(0, (4, 3)), alpha=0.8)
ax.text(103, fr["observed_decline_base_rate"] + 0.008, f"base rate {FACTS['base_rate']}",
        ha="right", va="bottom", fontsize=9, color=INK2)
ax.annotate(f"{FACTS['hgb_p50']}", xy=(50, m["hist-gbm"]["p_at_50_mean"]), xytext=(56, 0.885),
            fontsize=11, fontweight="bold", color=ACCENT,
            arrowprops=dict(arrowstyle="-", color=ACCENT, lw=1))
ax.annotate(f"{FACTS['rule_p50']}", xy=(50.5, m["baseline"]["p_at_50_mean"] + 0.004), xytext=(56, 0.605),
            fontsize=10, color=INK3, arrowprops=dict(arrowstyle="-", color=INK3, lw=1))
ax.set_xticks(ks)
ax.set_xlim(4, 108)
ax.set_ylim(0.38, 0.98)
ax.set_xlabel("K — pages reviewed")
ax.set_ylabel("precision@K")
head(ax, "Precision at K on client-grouped holdouts",
     "mean of 5 draws · bars at K=50 are the min–max across draws")
ax.legend(frameon=False, fontsize=8.8, loc="upper left", bbox_to_anchor=(0, -0.14),
          ncol=4, handlelength=2.2, columnspacing=1.8)
fig.tight_layout()
save(fig, "fig-model-vs-baseline.png")

# ── figure 2: the split is the experiment ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8.6, 4.3))
groups = ["Random page split", "Client-grouped split"]
p50 = [sc["before"]["P@50"], sc["after"]["P@50"]]
rocauc = [sc["before"]["AUC"], sc["after"]["AUC"]]
x = np.arange(len(groups))
w = 0.30
b1 = ax.bar(x - w / 2, p50, w, color=ACCENT, alpha=0.9, label="precision@50")
b2 = ax.bar(x + w / 2, rocauc, w, color=ACCENT2, alpha=0.45, label="ROC-AUC")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.015, f"{b.get_height():.3f}".rstrip("0").rstrip("."),
                ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.axhline(fr["observed_decline_base_rate"], color=INK3, lw=1, ls=(0, (4, 3)))
ax.text(0.5, fr["observed_decline_base_rate"] + 0.02, f"base rate {FACTS['base_rate']}",
        ha="center", va="bottom", fontsize=9, color=INK2)
ax.set_xticks(x)
ax.set_xticklabels(groups, fontsize=10.5)
ax.set_ylim(0, 1.12)
ax.set_ylabel("score")
ax.yaxis.grid(True)
ax.xaxis.grid(False)
head(ax, "The same model, two split designs",
     f"label ICC across clients {FACTS['icc']} · a random split lets the model meet the same client twice")
ax.legend(frameon=False, fontsize=9, loc="upper right")
fig.tight_layout()
save(fig, "fig-split-before-after.png")


CLIENT-GROUPED HOLDOUTS — base rate 51.5%, 5 seeds

                        ranker P@10 P@20  P@50 P@100 ROC-AUC PR-AUC
          Frozen rule baseline  0.6 0.62  0.54 0.556   0.554  0.499
           Logistic regression  0.6 0.69 0.748 0.804   0.656  0.611
   Gradient boosting (shipped) 0.84 0.79  0.82 0.806   0.643  0.591
Gradient boosting, no momentum 0.54 0.54 0.516 0.484   0.560  0.497

P@50 range: shipped 0.62–0.94 | rule 0.46–0.68
share of the lift from the momentum feature: 74%
ablation without log_momentum: P@50 0.516 — below the rule baseline it must beat


  wrote docs/assets/fig-model-vs-baseline.png  (79,361 bytes)


  wrote docs/assets/fig-split-before-after.png  (48,217 bytes)


WindowsPath('E:/AI/PE/flyrank-ml-internship/docs/assets/fig-split-before-after.png')

## 5. Limitations

*What this work cannot claim.*

The limits are not a disclaimer at the end; they are the boundary of the product. Each line below narrows what
the numbers in §4 are allowed to mean, and the page states them in the same words.


In [6]:
limits = [
    ("No causal claim", "nothing was assigned at random; no page was changed by this work"),
    ("No uplift estimate", "the click gap is a sizing heuristic, not a forecast of recovery"),
    ("No revenue framing", "the release has no revenue or CPC column, so value is out of scope"),
    ("One month pair", "March features, April label; stability over time is untested"),
    ("57.4% frame", f"{FACTS['march_pages']} March pages, {FACTS['frame_pages']} in frame — the long tail is out of scope"),
    ("Survivorship", f"{FACTS['dropped_no_april']} pages had no April row and were dropped, not labelled"),
    ("Unobserved interventions", "refreshes during April are indistinguishable from organic decline"),
    ("Nine clients per draw", f"P@50 {FACTS['hgb_p50_range']} vs rule {FACTS['rule_p50_range']} — ranges overlap at the edges"),
    ("Momentum is partly mechanical", "the label compares April to all of March, inflating the feature's measured value"),
    ("Modest rank quality", f"ROC-AUC {FACTS['hgb_auc']} orders a queue; it does not score a page"),
    ("A client-heavy head", f"{FACTS['client_concentration']} of the top 100 is one client; interleaving mitigates, not fixes"),
]
print("WHAT THIS WORK CANNOT CLAIM\n")
for claim, why in limits:
    print(f"  - {claim:30} {why}")

print("\nreceipt limitations carried forward verbatim:")
for src, items in [("w05_model_metrics.json", model["limitations"]), ("w06_validation_audit.json", audit["limitations"])]:
    for item in items:
        print(f"  [{src.split('_')[0]}] {item}")


WHAT THIS WORK CANNOT CLAIM

  - No causal claim                nothing was assigned at random; no page was changed by this work
  - No uplift estimate             the click gap is a sizing heuristic, not a forecast of recovery
  - No revenue framing             the release has no revenue or CPC column, so value is out of scope
  - One month pair                 March features, April label; stability over time is untested
  - 57.4% frame                    176,738 March pages, 101,441 in frame — the long tail is out of scope
  - Survivorship                   548 pages had no April row and were dropped, not labelled
  - Unobserved interventions       refreshes during April are indistinguishable from organic decline
  - Nine clients per draw          P@50 0.62–0.94 vs rule 0.46–0.68 — ranges overlap at the edges
  - Momentum is partly mechanical  the label compares April to all of March, inflating the feature's measured value
  - Modest rank quality            ROC-AUC 0.643 orders a que

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

A ranking is not a recommendation. This is the part a team would use: how the queue narrows, which reason code
prescribes which action, what an archetype says about a page, what the weekly batch costs, and what must never
happen without a human.

The split of labour is deliberate. **The model ranks** — it is the only thing that sees all eight features at
once. **The rules explain and prescribe** — deterministic, inspectable, and arguable. A 0.64-AUC ranker earns
the right to order a queue; it has not earned the right to decide what happens to a page.


In [7]:
disp = pd.DataFrame([
    ("actionable", th(play["dispositions"]["actionable"]), "a reason code fired and the page is worth a person's time"),
    ("watchlist", th(play["dispositions"]["watchlist"]), "scored, below the effort or expected-clicks floor"),
    ("no action", th(play["dispositions"]["no_action"]), "nothing fired: steady page, adequate CTR for its position"),
], columns=["disposition", "pages", "meaning"])
print("DISPOSITIONS\n")
print(disp.to_string(index=False))
print(f"\nactionable is {FACTS['actionable_share']} of the frame "
      f"({FACTS['frame_pages']} scored pages)")

ACTIONS = {
    "CTR_FAR_BELOW_PEERS": ("SNIPPET_REWRITE", "rewrite title and snippet, then measure"),
    "MOMENTUM_FADE":       ("REFRESH_BODY", "refresh the body and re-check intent"),
    "CTR_BELOW_PEERS":     ("SNIPPET_TEST", "test a snippet change first, escalate if the gap survives"),
    "ZERO_CLICKS":         ("INTENT_CHECK", "check intent mismatch before writing anything"),
}
reasons = pd.DataFrame([
    (code, n, ACTIONS[code][0], ACTIONS[code][1])
    for code, n in play["reason_mix_actionable"].items()
], columns=["reason code", "pages", "action", "what a person does"]).sort_values("pages", ascending=False)
print("\n\nREASON CODE -> ACTION (one primary code per actionable page)\n")
print(reasons.to_string(index=False))

arch_tbl = pd.DataFrame([
    (name, th(a["pages"]), pct(a["observed_decline_rate"]),
     "—" if a["mean_click_gap"] is None else f"{a['mean_click_gap']:.2f}", a["mapped_action"])
    for name, a in play["archetypes"].items()
], columns=["archetype", "pages", "declined", "click gap", "what to do"])
print("\n\nARCHETYPES (decline rate is the observed April outcome of that band)\n")
print(arch_tbl.to_string(index=False))
print(f"\nspread across archetypes: {pct(min(a['observed_decline_rate'] for a in play['archetypes'].values()))} "
      f"to {pct(max(a['observed_decline_rate'] for a in play['archetypes'].values()))} "
      f"around a {FACTS['base_rate']} base rate")
print("  archetype routes the work; the model and the codes decide the order")

print(f"\n\nWEEKLY BATCH: {FACTS['batch_actions']} actions | {FACTS['batch_clients']} clients | "
      f"{FACTS['batch_hours']} hours | {FACTS['batch_clicks']} clicks of measured gap")
for action, n in sorted(wb["action_mix"].items(), key=lambda kv: -kv[1]):
    print(f"  {action:16} {n}")
print("  interleaved by client, capped at 3 of 50 — the model's global head is one client's backlog")

print("\nNO-GO LIST (never automated)\n")
for item in play["no_go"]:
    print(f"  x {item}")

# ── figure 4: the funnel ──────────────────────────────────────────────────
stages = [
    ("March pages", fr["march_pages_in_release"], "any impressions in the release"),
    ("Scored", fr["pages_described_by_features"], "at least 100 March impressions"),
    ("Actionable", play["dispositions"]["actionable"], "reason code fired, above both gates"),
    ("Weekly batch", wb["actions"], "interleaved, 3 per client maximum"),
]
fig, ax = plt.subplots(figsize=(9.6, 4.1))
ys = np.arange(len(stages))[::-1]
vals = [s[1] for s in stages]
ax.barh(ys, vals, height=0.52, color=ACCENT, alpha=0.9)
ax.barh(ys[-1], vals[-1], height=0.52, color=CLAY)
ax.plot([vals[-1], vals[-1]], [ys[-1] - 0.26, ys[-1] + 0.26], color=CLAY, lw=3, solid_capstyle="butt")
for y, (label, v, note) in zip(ys, stages):
    ax.text(v + 2600, y + 0.02, f"{v:,}", va="bottom", ha="left", fontsize=11, fontweight="bold")
    ax.text(v + 2600, y - 0.22, note, va="top", ha="left", fontsize=8.8, color=INK2)
ax.set_yticks(ys)
ax.set_yticklabels([s[0] for s in stages], fontsize=10.5)
ax.set_xlim(0, 205000)
ax.set_xlabel("pages")
ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
ax.yaxis.grid(False)
head(ax, "From 176,738 pages to fifty actions",
     f"1 in {fr['march_pages_in_release'] // wb['actions']:,} pages reaches the weekly batch")
fig.tight_layout()
save(fig, "fig-frame-funnel.png")


DISPOSITIONS

disposition  pages                                                   meaning
 actionable  8,395 a reason code fired and the page is worth a person's time
  watchlist 81,137         scored, below the effort or expected-clicks floor
  no action 11,909 nothing fired: steady page, adequate CTR for its position

actionable is 8.3% of the frame (101,441 scored pages)


REASON CODE -> ACTION (one primary code per actionable page)

        reason code  pages          action                                        what a person does
CTR_FAR_BELOW_PEERS   3493 SNIPPET_REWRITE                   rewrite title and snippet, then measure
      MOMENTUM_FADE   3084    REFRESH_BODY                      refresh the body and re-check intent
    CTR_BELOW_PEERS   1753    SNIPPET_TEST test a snippet change first, escalate if the gap survives
        ZERO_CLICKS     65    INTENT_CHECK             check intent mismatch before writing anything


ARCHETYPES (decline rate is the observed April outc

  wrote docs/assets/fig-frame-funnel.png  (62,553 bytes)


WindowsPath('E:/AI/PE/flyrank-ml-internship/docs/assets/fig-frame-funnel.png')

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Four figures, one social card, one facts file, and then the audit that makes the page trustworthy: every number
the page states is re-read from the HTML and compared with the receipt that produced it. If a number drifts,
this cell fails.


In [8]:
# ── figure 3: momentum bands vs observed outcome ──────────────────────────
bands = list(dec["by_momentum_band"].items())
pretty = {"falling hard (<-0.70)": "falling hard\n< \u22120.70",
          "falling (-0.70..-0.35)": "falling\n\u22120.70 to \u22120.35",
          "soft (-0.35..0)": "soft\n\u22120.35 to 0",
          "steady (0..0.35)": "steady\n0 to 0.35",
          "rising (>0.35)": "rising\n> 0.35"}
labels = [f"{pretty.get(b[0], b[0])}\nn={b[1]['pages']:,}" for b in bands]
rates = [b[1]["observed_decline_rate"] for b in bands]

fig, ax = plt.subplots(figsize=(9.6, 4.6))
xs = np.arange(len(bands))
alphas = np.linspace(0.95, 0.35, len(bands))   # darkest = worst outcome
ax.bar(xs, rates, width=0.62, color=[matplotlib.colors.to_rgba(ACCENT, a) for a in alphas])
for x, r in zip(xs, rates):
    ax.text(x, r + 0.018, f"{r*100:.1f}%", ha="center", va="bottom", fontsize=10.5, fontweight="bold",
            path_effects=[pe.withStroke(linewidth=3.2, foreground=PAPER_C)])
ax.axhline(fr["observed_decline_base_rate"], color=CLAY, lw=1.4, ls=(0, (5, 3)))
ax.text(len(bands) - 0.45, fr["observed_decline_base_rate"] - 0.022, f"base rate {FACTS['base_rate']}",
        ha="right", va="top", fontsize=9, color=CLAY)
ax.set_xticks(xs)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylim(0, 1.0)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0, decimals=0))
ax.set_ylabel("observed April decline rate")
ax.yaxis.grid(True)
ax.xaxis.grid(False)
head(ax, "Momentum in March, outcome in April",
     "all scored pages, one month pair — an association, not a lifecycle curve")
fig.tight_layout()
save(fig, "fig-momentum-decay.png")

# ── social card (1200 x 630) ──────────────────────────────────────────────
import textwrap
fig = plt.figure(figsize=(12, 6.3), dpi=100)
fig.patch.set_facecolor("#131417")
ax = fig.add_axes([0, 0, 1, 1]); ax.axis("off"); ax.set_facecolor("#131417")
ax.text(0.06, 0.89, "FLYRANK ML INTERNSHIP  ·  LANE 4  ·  MARCH–APRIL 2026",
        fontsize=11, color="#8fbde8", family="DejaVu Sans", fontweight="bold")
ax.text(0.06, 0.72, "Which pages should\nyou review first?", fontsize=44, color="#f6f4ef",
        family="DejaVu Sans", fontweight="bold", linespacing=1.08, va="top")
ax.text(0.06, 0.44, "An early-warning queue for organic click decline,\nvalidated with clients held out.",
        fontsize=16, color="#b3b0a9", family="DejaVu Sans", linespacing=1.45, va="top")
for i, (value, label) in enumerate([(FACTS["hgb_p50"], "model P@50"), (FACTS["rule_p50"], "frozen rule"),
                                    (FACTS["base_rate"], "base rate")]):
    x = 0.06 + i * 0.22
    ax.text(x, 0.20, value, fontsize=30, color="#f6f4ef" if i == 0 else "#8b8880",
            family="DejaVu Sans", fontweight="bold", va="top")
    ax.text(x, 0.09, label, fontsize=11, color="#8b8880", family="DejaVu Sans", va="top")
ax.text(0.94, 0.09, "louai-mohamed.github.io/flyrank-ml-internship",
        fontsize=10.5, color="#8fbde8", family="DejaVu Sans", ha="right", va="top")
fig.savefig(ASSETS / "social-preview.png", dpi=100, facecolor=fig.get_facecolor())
plt.close(fig)
print(f"  wrote docs/assets/social-preview.png  ({(ASSETS / 'social-preview.png').stat().st_size:,} bytes)")

# ── the facts file: what the page is allowed to say ───────────────────────
(ASSETS / "paper-facts.json").write_text(
    json.dumps(FACTS, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
print(f"  wrote docs/assets/paper-facts.json  ({len(FACTS)} facts)")


  wrote docs/assets/fig-momentum-decay.png  (67,754 bytes)
  wrote docs/assets/social-preview.png  (53,398 bytes)
  wrote docs/assets/paper-facts.json  (55 facts)


### The audit

The page is static HTML, so the numbers in it could drift from the receipts without anyone noticing. This cell
closes that hole: it parses the page, finds every `<span data-fact="…">`, and compares the rendered text with the
value generated from the receipt. It also checks the paper's own contract — nine sections, the data credit, the
figures, the links, and the absence of anything private.


In [9]:
html = PAPER.read_text(encoding="utf-8")
results = []

def check(name, ok, detail=""):
    results.append((name, bool(ok), detail))
    return ok

# 1. the nine required sections, in the required order
required = ["abstract", "introduction", "data", "method", "results", "limits", "playbook", "repro", "credit"]
present = re.findall(r'<section[^>]*\bid="([^"]+)"', html)
check("all 9 required sections present", all(s in present for s in required),
      f"missing: {[s for s in required if s not in present] or 'none'}")
order_ok = all(html.index(f'id="{a}"') < html.index(f'id="{b}"') for a, b in zip(required, required[1:]))
check("sections appear in reading order", order_ok)

# 2. abstract first, data credit last (the two sections people forget)
check("abstract is the first block after the masthead", html.index('id="abstract"') < html.index('id="introduction"'))
check("data credit links to flyrank.ai", 'href="https://flyrank.ai"' in html)
check("credit names the dataset", "FlyRank ML Internship dataset" in html)

# 3. every number on the page matches its receipt
spans = re.findall(r'data-fact="([^"]+)">([^<]*)<', html)
bad = [(k, v, FACTS.get(k)) for k, v in spans if FACTS.get(k) != v]
check(f"{len(spans)} data-fact spans match the receipts ({len(set(k for k, _ in spans))} unique)", not bad,
      f"{len(bad)} mismatch(es): {bad[:3]}" if bad else "")
unused = sorted(set(FACTS) - set(k for k, _ in spans))
check("every generated fact is used by the page", not unused, f"unused: {unused}" if unused else "")

# 4. figures and assets exist, and are referenced relatively
imgs = re.findall(r'<img[^>]*src="([^"]+)"', html)
missing = [i for i in imgs if not (DOCS / i).exists()]
check(f"all {len(imgs)} figures exist on disk", not missing, f"missing: {missing}" if missing else "")
check("figure paths are relative", all(not i.startswith(("/", "http")) for i in imgs))
check("every figure has alt text", len(re.findall(r'<img[^>]*\balt="[^"]+"', html)) == len(imgs))

# 5. public safety
urls = sorted(set(re.findall(r'https?://[^\s"\'<>)]+', html)))
allowed = ("https://github.com/Louai-Mohamed", "https://raw.githubusercontent.com/Louai-Mohamed",
           "https://louai-mohamed.github.io", "https://flyrank.ai", "https://huggingface.co",
           "https://creativecommons.org", "https://scikit-learn.org", "https://schema.org", "http://www.w3.org")
offenders = [u for u in urls if not u.startswith(allowed)]
check(f"{len(urls)} external links, all allow-listed", not offenders, f"unexpected: {offenders}" if offenders else "")
check("no email addresses", not re.search(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", html))
check("no external scripts or web fonts", "<script src=" not in html and "fonts.googleapis" not in html)
check("no client or content identifiers", not re.search(r"(client|content)_hash_id\s*[=:]\s*[\"']?[0-9a-f]{6}", html))

# 6. the submission file points at this page
url_file = ROOT / "submission" / "paper_url.txt"
if url_file.exists():
    target = url_file.read_text(encoding="utf-8").strip()
    check("submission/paper_url.txt holds exactly one https URL",
          target.startswith("https://") and "\n" not in target.strip(), target)
    check("paper_url.txt points at this page", target.rstrip("/") == "https://louai-mohamed.github.io/flyrank-ml-internship")
    try:
        import urllib.request
        with urllib.request.urlopen(target, timeout=15) as r:
            body = r.read().decode("utf-8", "replace")
        check(f"the live page answers {r.status} and carries the credit", r.status == 200 and "flyrank.ai" in body)
    except Exception as exc:
        print(f"  (live check skipped: {exc})")

print("PAGE AUDIT\n")
for name, ok, detail in results:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f"  — {detail}" if detail else ""))
failed = [r for r in results if not r[1]]
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed")
if failed:
    raise AssertionError("page and receipts disagree: " + "; ".join(r[0] for r in failed))


PAGE AUDIT

  PASS  all 9 required sections present  — missing: none
  PASS  sections appear in reading order
  PASS  abstract is the first block after the masthead
  PASS  data credit links to flyrank.ai
  PASS  credit names the dataset
  PASS  88 data-fact spans match the receipts (55 unique)
  PASS  every generated fact is used by the page
  PASS  all 4 figures exist on disk
  PASS  figure paths are relative
  PASS  every figure has alt text
  PASS  13 external links, all allow-listed
  PASS  no email addresses
  PASS  no external scripts or web fonts
  PASS  no client or content identifiers
  PASS  submission/paper_url.txt holds exactly one https URL  — https://louai-mohamed.github.io/flyrank-ml-internship/
  PASS  paper_url.txt points at this page
  PASS  the live page answers 200 and carries the credit

17/17 checks passed


## 8. Five-minute demo

*The showcase outline — question, method, one chart, one honest result, one recommendation.*

The Week-8 showcase is optional; if you take the five minutes, this is the run of show. It is built around
**one chart, one table, one number**, because a five-minute talk that shows six charts is a five-minute list of
files.

| Time | Say this | Show |
|---|---|---|
| 0:00–0:30 | **The decision.** A content team has 101,441 pages and room for fifty reviews a week. FlyRank already flags pages with hand-written rules — health score, quick-win tags, needs-attention flags. The question is whether a model can order that queue better than the rule. | Title slide, nothing else |
| 0:30–1:15 | **The data.** One pseudonymized release: 78.8 million daily rows of real search data, March features, April label, 43 clients. Nothing synthetic, nothing scraped, no client named. | Table 1 — the windows |
| 1:15–2:15 | **The method, and the one choice that matters.** Label: April impressions below 80% of March. Eight features knowable on 31 March. And the split holds out whole clients, because a random page split lets a model memorise client identity and call it skill. | Table 4 — two split designs |
| 2:15–3:15 | **The chart.** Precision at K across five client-grouped draws. The model and the rule separate exactly where the decision happens: the first fifty rows. | **Figure 1 — the only chart** |
| 3:15–4:00 | **The honest result.** 0.82 against 0.54 on the identical split, at a 51.5% base rate. Then the number that keeps it honest: drop the momentum feature and the model falls to 0.516 — below the rule it replaced. 74% of the gain is one feature, not cleverness. | Figure 1, pointing at the ablation line |
| 4:00–4:45 | **The recommendation.** The model ranks; deterministic reason codes prescribe. 8,395 actionable pages become a 50-action weekly batch across 25 clients, and every row ships with `auto_apply_allowed = False`. | Table 7 — dispositions |
| 4:45–5:00 | **The close.** It orders work that was going to happen anyway. It does not decide what happens to a page. | The paper URL |

**Backup, not opener:** Figure 4 (the funnel from 176,738 pages to fifty actions) if someone asks how the queue
narrows. Figure 3 (momentum vs observed outcome) if someone asks whether the signal is monotone.

**If someone asks — 20-second answers**

- *"Isn't an AUC of 0.64 weak?"* — For scoring one page, yes, and §5 says so. For ordering a queue it lifts the top
  fifty from 0.54 to 0.82 against a 51.5% base rate. The product is a ranking, so that is the metric that matters.
- *"Why not ordinary cross-validation?"* — The label's ICC across clients is 0.133. A random page split scored 0.992
  and measured client memorisation; whole-client holdout scored 0.82 and measured something that might transfer.
- *"Did you check for leakage?"* — Three ways, including a deliberate injection: give the model the label's own
  ingredient and AUC goes to 0.999, so the detector demonstrably fires. The shipped features stay at 0.617.
- *"What would you do with another month of data?"* — Re-run the same design on a different month pair before
  trusting any threshold, and measure whether a refreshed page actually recovers. Neither is possible in this
  design, which is the honest answer.


## 9. Two shareable cuts

*The same work, retold. Same evidence, different door.*

### Cut 1 — social post, on the methodology

> I spent a month ranking 101,441 content pages by whether they would lose search clicks the next month.
>
> The model hit precision@50 of 0.82. The hand-written rule it replaced scored 0.54. Base rate: 51.5%.
>
> But the number I would actually defend is this one: with a random page split, the same model scored 0.99.
>
> Not because it got better. Because it had already met the clients whose pages it was "predicting". Pages from
> one client share a template, a tracking setup, and a seasonality. Let the model see the client in training and it
> memorises the client, then reports that as skill.
>
> Hold the clients out and the score falls to 0.82 — lower, and the first number you can hand to someone who has
> to deploy it.
>
> 0.99 vs 0.82 is not a modelling story. It is a validation story, and validation is what decides whether a result
> survives contact with production.
>
> Paper and notebooks: https://louai-mohamed.github.io/flyrank-ml-internship/

**Short variant (fits a single post):**

> Same model, two splits. Random pages: 0.99 precision@50. Whole clients held out: 0.82. Nothing about the model
> changed — the first number was memorising clients and calling it skill. Validation is the experiment.
> https://louai-mohamed.github.io/flyrank-ml-internship/

### Cut 2 — employer-facing, three sentences

**What I built.** A ranking model that tells a content team which pages to review first, plus the deterministic
rules that decide what to do with each one — the model orders, the rules prescribe, a human approves.

**On what data.** A pseudonymized production search warehouse: 78.8 million daily rows of real Google Search
Console and Analytics data, reduced to 101,441 content pages across 43 clients, with whole clients held out of
training and the label taken from the following month.

**What it showed.** On that honest split, precision@50 went from 0.54 for a frozen hand-written rule to 0.82
(0.62–0.94 across five draws) — and an ablation showed 74% of the gain came from a single feature, momentum,
rather than from model complexity.

*One line if you only get one:* I built a ranked review queue on 79 million rows of real search data, and the
hardest part was proving the number was real.


## Self-check

Before submitting, each line below is true — checked, not assumed.

- [x] **Every section is filled** — markdown thinking *and* the code that backs it (1–9 above)
- [x] **The notebook runs top to bottom with no errors** — executed, outputs committed
- [x] **No client names, URLs, or private queries anywhere** — the audit in §7 fails the build if one appears
- [x] **Claims use careful words** — observed, measured, directional, decision-support; the causal claim is explicitly refused in §5
- [x] **Committed under `work/notebooks/`** — and the paper is deployed from `docs/`
- [x] **The deployed paper has all 9 sections** — abstract at the top, data credit at the bottom, both audited in §7
- [x] **The paper ties back to the FlyRank content problem** — §1 names the product, the decay problem, and the Lane 4 brief
- [x] **`submission/paper_url.txt` holds the exact deployed URL** — one line, checked against the live page
- [x] **The page's numbers cannot drift** — 88 `data-fact` spans re-read from the HTML and compared with the receipts
- [x] **The demo and the two cuts are ready** — §8 and §9 above, built from the same receipts as the paper
